# Fase III — Entrenamiento BiLSTM-CRF
Pipeline completo para el entrenamiento, evaluación y predicción usando un modelo BiLSTM con una capa Conditional Random Field (CRF).

In [1]:
!pip install torch torchvision torchaudio
!pip install pandas scikit-learn
!pip install pytorch-crf

  Using cached torch-2.11.0-cp311-cp311-win_amd64.whl.metadata (29 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ----------------------- ---------------- 2.4/4.0 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 12.5 MB/s  0:00:00
Using cached torch-2.11.0-cp311-cp311-win_amd64.whl (114.5 MB)

   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
  Attempting uninstall: torch
   ---------------------------------------- 0/3 [torchaudio]
    Found existing installation: torch 2.9.1
   ---------------------------------------- 0/3 [torchaudio]
   ------------- -------------------------- 1/3 [torch]
   ------------- -------------------------- 1/3 [torch]
   ------------- ------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\jfleo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\jfleo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\jfleo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### 1. Importaciones y Configuración del Dispositivo

In [3]:
import os
import copy
import time
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torchcrf import CRF

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

Dispositivo: cpu


### 2. Definición del Modelo BiLSTM-CRF

In [4]:
class BiLSTMCRFTagger(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim, hidden_dim, pad_idx=0):
        super(BiLSTMCRFTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, tagset_size)
        self.crf = CRF(tagset_size, batch_first=True)

    def _get_emissions(self, x):
        embeds = self.embedding(x)
        lstm_out, _ = self.lstm(embeds)
        emissions = self.fc(lstm_out)
        return emissions

    def forward(self, x, tags, mask):
        emissions = self._get_emissions(x)
        loss = -self.crf(emissions, tags, mask=mask, reduction='mean')
        return loss

    def decode(self, x, mask):
        emissions = self._get_emissions(x)
        return self.crf.decode(emissions, mask=mask)

### 3. Funciones de Entrenamiento y Evaluación

In [5]:
def train_model(model, train_loader, val_loader, optimizer, device, epochs=50, patience=5):
    model = model.to(device)
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            mask = batch["attention_mask"].to(device).bool()

            optimizer.zero_grad()
            loss = model(input_ids, labels, mask)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        avg_train = total_train_loss / len(train_loader)
        train_losses.append(avg_train)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                labels = batch["labels"].to(device)
                mask = batch["attention_mask"].to(device).bool()
                loss = model(input_ids, labels, mask)
                total_val_loss += loss.item()

        avg_val = total_val_loss / len(val_loader)
        val_losses.append(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
            marker = " *"
        else:
            patience_counter += 1
            marker = f" (patience {patience_counter}/{patience})"
            if patience_counter >= patience:
                print(f"  Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}{marker}")
                print(f"  Early stopping en epoch {epoch+1}")
                break

        print(f"  Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}{marker}")

    model.load_state_dict(best_model_state)
    return model, train_losses, val_losses, best_val_loss

def evaluate_model(model, test_loader, tag2idx, device):
    idx2tag = {v: k for k, v in tag2idx.items()}
    all_preds, all_labels = [], []
    model.eval()
    model = model.to(device)

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            mask = batch["attention_mask"].to(device).bool()

            pred_sequences = model.decode(input_ids, mask)

            for i in range(len(pred_sequences)):
                pred_tags = pred_sequences[i]
                true_tags = labels[i][mask[i]].cpu().numpy()
                all_preds.extend([idx2tag.get(p, "<UNK>") for p in pred_tags])
                all_labels.extend([idx2tag.get(t, "<UNK>") for t in true_tags])

    return {
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average='macro', zero_division=0),
        "weighted_f1": f1_score(all_labels, all_preds, average='weighted', zero_division=0),
        "report": classification_report(all_labels, all_preds, zero_division=0)
    }

### 4. Funciones de Inferencia (Predicción)

In [6]:
def predict_sentence(sentence, model, word2idx, tag2idx, device):
    idx2tag = {v: k for k, v in tag2idx.items()}
    words = sentence.strip().split()
    if not words: return []

    encoded = [word2idx.get(w, word2idx.get("<UNK>", 1)) for w in words]
    input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)
    mask = torch.ones(1, len(words), dtype=torch.bool).to(device)

    model.eval()
    with torch.no_grad():
        pred_indices = model.decode(input_tensor, mask)[0]

    pred_tags = [idx2tag.get(idx, "<UNK>") for idx in pred_indices]
    return list(zip(words, pred_tags))

def print_prediction(sentence, model, word2idx, tag2idx, device, label):
    preds = predict_sentence(sentence, model, word2idx, tag2idx, device)
    print(f"\n--- {label} ---")
    for word, tag in preds:
        print(f"  {word:20s} -> {tag}")

### 5. Grid Search

In [7]:
def run_grid_search(train_ds, val_ds, vocab_size, tagset_size, dataset_name, device, smoke_test=False):
    if smoke_test:
        batch_sizes, optimizers_config = [64], [{"name": "Adam", "lr": 0.001}]
        embedding_dims, hidden_dims = [100], [128]
        epochs, patience = 2, 1
        print(f"  [SMOKE TEST] 1 config rápida.")
    else:
        batch_sizes = [16, 32, 64]
        optimizers_config = [
            {"name": "Adam", "lr": 0.001},
            {"name": "SGD", "lr": 0.01, "momentum": 0.9}
        ]
        embedding_dims = [100, 300]
        hidden_dims = [128, 256]
        epochs, patience = 50, 5

    results = []
    best_val_loss = float('inf')
    best_model, best_config = None, None

    total = len(batch_sizes) * len(optimizers_config) * len(embedding_dims) * len(hidden_dims)
    combo = 0

    for bs in batch_sizes:
        train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False)

        for opt_cfg in optimizers_config:
            for emb_dim in embedding_dims:
                for hid_dim in hidden_dims:
                    combo += 1
                    print(f"\n[{dataset_name}] {combo}/{total}: bs={bs}, opt={opt_cfg['name']}, emb={emb_dim}, hid={hid_dim}")

                    model = BiLSTMCRFTagger(vocab_size, tagset_size, emb_dim, hid_dim)

                    if opt_cfg["name"] == "Adam":
                        optimizer = optim.Adam(model.parameters(), lr=opt_cfg["lr"])
                    else:
                        optimizer = optim.SGD(model.parameters(), lr=opt_cfg["lr"], momentum=opt_cfg["momentum"])

                    start = time.time()
                    trained_model, t_losses, v_losses, val_loss = train_model(
                        model, train_loader, val_loader, optimizer, device, epochs=epochs, patience=patience
                    )
                    elapsed = time.time() - start

                    config = {
                        "batch_size": bs, "optimizer": opt_cfg["name"],
                        "embedding_dim": emb_dim, "hidden_dim": hid_dim,
                        "best_val_loss": round(val_loss, 4), "epochs_run": len(t_losses), "time_s": round(elapsed, 1)
                    }
                    results.append(config)
                    print(f"  -> Val Loss: {val_loss:.4f} | Epochs: {len(t_losses)} | Tiempo: {elapsed:.1f}s")

                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        best_model = copy.deepcopy(trained_model)
                        best_config = config.copy()

                    del model, trained_model, optimizer
                    if device.type == 'cuda': torch.cuda.empty_cache()

    print(f"\nMEJOR CONFIG [{dataset_name}]: {best_config}")
    return best_model, best_config, results

### 6. Ejecución Principal (Dataset, Entrenamiento y Pruebas)

In [ ]:
# --- CONFIGURACIÓN ---
SMOKE_TEST = False # Cambia a True para una prueba ultra rápida

class POSDataset(Dataset):
    def __init__(self, inputs, labels, masks):
        self.inputs = inputs if isinstance(inputs, torch.Tensor) else torch.tensor(inputs, dtype=torch.long)
        self.labels = labels if isinstance(labels, torch.Tensor) else torch.tensor(labels, dtype=torch.long)
        self.masks  = masks  if isinstance(masks,  torch.Tensor) else torch.tensor(masks,  dtype=torch.long)
    def __len__(self): return len(self.inputs)
    def __getitem__(self, idx): return {"input_ids": self.inputs[idx], "labels": self.labels[idx], "attention_mask": self.masks[idx]}

# -- RUTAS (Asumiendo que el notebook se ejecuta en la raíz del proyecto) --
processed_dir = os.path.join("outputs", "processed")
models_dir = os.path.join("outputs", "models")
os.makedirs(models_dir, exist_ok=True)

print("Cargando Ancora...")
ancora = torch.load(os.path.join(processed_dir, "ancora_data.pt"), weights_only=False)
train_ds_a = POSDataset(ancora["train_inputs"], ancora["train_labels"], ancora["train_masks"])
val_ds_a = POSDataset(ancora["val_inputs"], ancora["val_labels"], ancora["val_masks"])
test_ds_a = POSDataset(ancora["test_inputs"], ancora["test_labels"], ancora["test_masks"])

print("Cargando CoNLL2002...")
conll = torch.load(os.path.join(processed_dir, "conll_data.pt"), weights_only=False)
train_ds_c = POSDataset(conll["train_inputs"], conll["train_labels"], conll["train_masks"])
val_ds_c = POSDataset(conll["val_inputs"], conll["val_labels"], conll["val_masks"])
test_ds_c = POSDataset(conll["test_inputs"], conll["test_labels"], conll["test_masks"])

# -- Entrenamiento Ancora --
print("\n" + "="*40 + "\nENTRENANDO ANCORA (BiLSTM-CRF)\n" + "="*40)
best_model_a, best_config_a, results_a = run_grid_search(
    train_ds_a, val_ds_a, len(ancora["word2idx"]), len(ancora["tag2idx"]), "Ancora", device, smoke_test=SMOKE_TEST
)
eval_a = evaluate_model(best_model_a, DataLoader(test_ds_a, batch_size=64), ancora["tag2idx"], device)
torch.save({"model_state_dict": best_model_a.state_dict(), "best_config": best_config_a}, os.path.join(models_dir, "bilstm_crf_ancora.pt"))

# -- Entrenamiento CoNLL --
print("\n" + "="*40 + "\nENTRENANDO CONLL (BiLSTM-CRF)\n" + "="*40)
best_model_c, best_config_c, results_c = run_grid_search(
    train_ds_c, val_ds_c, len(conll["word2idx"]), len(conll["tag2idx"]), "CoNLL2002", device, smoke_test=SMOKE_TEST
)
eval_c = evaluate_model(best_model_c, DataLoader(test_ds_c, batch_size=64), conll["tag2idx"], device)
torch.save({"model_state_dict": best_model_c.state_dict(), "best_config": best_config_c}, os.path.join(models_dir, "bilstm_crf_conll.pt"))

# -- Resumen Final --
print("\n" + "=" * 50 + "\nRESUMEN FINAL BiLSTM-CRF\n" + "=" * 50)
print(f"ANCORA -> Accuracy: {eval_a['accuracy']:.4f} | F1 Macro: {eval_a['macro_f1']:.4f}")
print(f"CONLL  -> Accuracy: {eval_c['accuracy']:.4f} | F1 Macro: {eval_c['macro_f1']:.4f}")

test_sentence = "El modelo con CRF identifica muy bien las secuencias"
print_prediction(test_sentence, best_model_a, ancora["word2idx"], ancora["tag2idx"], device, "Predicción Ancora")


Cargando Ancora...
Cargando CoNLL2002...

ENTRENANDO ANCORA (BiLSTM-CRF)

[Ancora] 1/24: bs=16, opt=Adam, emb=100, hid=128
  Epoch 01/50 | Train Loss: 18.3967 | Val Loss: 9.6725 *
  Epoch 02/50 | Train Loss: 7.1083 | Val Loss: 6.3597 *
  Epoch 03/50 | Train Loss: 4.2085 | Val Loss: 4.9570 *
  Epoch 04/50 | Train Loss: 2.5581 | Val Loss: 4.5725 *
  Epoch 05/50 | Train Loss: 1.4713 | Val Loss: 4.3400 *
